# A Parse Table for a Shift-Reduce Parser

This notebook contains the parse table that is needed for a *shift reduce parser* that parses the following grammar:
$$
  \begin{eqnarray*}
  \mathrm{expr}        & \rightarrow & \mathrm{expr}\;\;\texttt{'+'}\;\;\mathrm{product}   \\
                       & \mid        & \mathrm{expr}\;\;\texttt{'-'}\;\;\mathrm{product}   \\
                       & \mid        & \mathrm{product}                                    \\[0.2cm]
  \mathrm{product}     & \rightarrow & \mathrm{product}\;\;\texttt{'*'}\;\;\mathrm{factor} \\
                       & \mid        & \mathrm{product}\;\;\texttt{'/'}\;\;\mathrm{factor} \\
                       & \mid        & \mathrm{factor}                                     \\[0.2cm]
  \mathrm{factor}      & \rightarrow & \texttt{'('} \;\;\mathrm{expr} \;\;\texttt{')'}     \\
                       & \mid        & \texttt{NUMBER} 
  \end{eqnarray*}
$$

The notebooks `01-Shift-Reduce-Parser-Pure.ipynb` and `02-Shift-Reduce-Parser.ipynb` load this notebook via `%run`.
Since `%run` copies every top-level name of this notebook into the loading notebook, this notebook only defines
the types and the tables that are needed by the parser.

## The Grammar Rules

We represent variables, tokens, and states as strings.  The states are named `'s0'`, `'s1'`, $\cdots$, `'s15'`.
A *marked rule* is a grammar rule with a dot `•` somewhere in its body.  Marked rules are also represented as
strings, e.g. `'e → e • "+" p'`.  A *symbol* is either a variable or a token.

In [ ]:
Variable = str
Token    = str
Symbol   = Variable | Token
State    = str
MarkedRule = str

A grammar rule $A \rightarrow X_1 \cdots X_n$ is represented as the pair `(A, (X1, ..., Xn))`, i.e. the body
of a rule is a tuple of symbols.

In [ ]:
Rule = tuple[Variable, tuple[Symbol, ...]]

Below, we define the *grammar rules* $r_1$, $\cdots$, $r_8$.  In order to keep the tables short, we abbreviate
the variables $\mathrm{expr}$, $\mathrm{product}$, and $\mathrm{factor}$ as `e`, `p`, and `f`.  The rules are
numbered in the same way as in the lecture notes.

In [ ]:
r1: Rule = ('e', ('e', '+', 'p'))
r2: Rule = ('e', ('e', '-', 'p'))
r3: Rule = ('e', ('p',))

r4: Rule = ('p', ('p', '*', 'f'))
r5: Rule = ('p', ('p', '/', 'f'))
r6: Rule = ('p', ('f',))

r7: Rule = ('f', ('(', 'e', ')'))
r8: Rule = ('f', ('NUMBER',))

## The Action Table

Next, we define the *action table* as a dictionary.  It encodes the function
$$\texttt{action}: Q \times T \rightarrow \texttt{Action},$$
which takes the state $q$ on top of the stack of states and the next token $t$ and tells the parser what to do.
There are four kinds of actions:
- `('shift', s)`, where `s` is a state:  The token $t$ is pushed onto the stack of symbols and the state `s` is
  pushed onto the stack of states.
- `('reduce', r)`, where `r` is a grammar rule:  The symbol stack is reduced with the rule `r`.
- `'accept'`:  The input has been parsed successfully.
- `'error'`:  The input contains a syntax error.

Hence, the type `Action` is defined as follows.

In [ ]:
Action = str | tuple[str, State] | tuple[str, Rule]

The keys of the dictionary `actionTable` are pairs of the form `(state, token)`.  The token `'$'` denotes the
end of the input.  In order to keep the table short, the action `'error'` is not stored:  For every pair that is
not a key of `actionTable`, the parser looks up the action `'error'`.

The table has been computed with the algorithm for SLR parsers that is discussed in the lecture notes.
For example, the state `s3` contains the marked rule `p → f •`.  Therefore, the parser reduces with the rule
$r_6 = (\mathrm{product} \rightarrow \mathrm{factor})$ if the next token is an element of
$$\mathrm{Follow}(\mathrm{product}) = \{ \texttt{'+'}, \texttt{'-'}, \texttt{'*'}, \texttt{'/'}, \texttt{')'}, \texttt{'\$'} \}.$$
The state `s1` is the only state where the parser accepts:  It contains the marked rule `ŝ → e • $`, so if the
next token is `'$'`, the whole input has been reduced to an expression.

In [ ]:
actionTable: dict[tuple[State, Token], Action] = {}

actionTable['s0', '('  ]    = ('shift', 's4')
actionTable['s0', 'NUMBER'] = ('shift', 's5')

actionTable['s1', '$'] = 'accept'
actionTable['s1', '+'] = ('shift', 's6')
actionTable['s1', '-'] = ('shift', 's7')

actionTable['s2', '$'] = ('reduce', r3)
actionTable['s2', '+'] = ('reduce', r3)
actionTable['s2', '-'] = ('reduce', r3)
actionTable['s2', '*'] = ('shift', 's8')
actionTable['s2', '/'] = ('shift', 's9')
actionTable['s2', ')'] = ('reduce', r3)

actionTable['s3', '$'] = ('reduce', r6)
actionTable['s3', '+'] = ('reduce', r6)
actionTable['s3', '-'] = ('reduce', r6)
actionTable['s3', '*'] = ('reduce', r6)
actionTable['s3', '/'] = ('reduce', r6)
actionTable['s3', ')'] = ('reduce', r6)

actionTable['s4', '('     ] = ('shift', 's4')
actionTable['s4', 'NUMBER'] = ('shift', 's5')

actionTable['s5', '$'] = ('reduce', r8)
actionTable['s5', '+'] = ('reduce', r8)
actionTable['s5', '-'] = ('reduce', r8)
actionTable['s5', '*'] = ('reduce', r8)
actionTable['s5', '/'] = ('reduce', r8)
actionTable['s5', ')'] = ('reduce', r8)

actionTable['s6', '('     ] = ('shift', 's4')
actionTable['s6', 'NUMBER'] = ('shift', 's5')

actionTable['s7', '('     ] = ('shift', 's4')
actionTable['s7', 'NUMBER'] = ('shift', 's5')

actionTable['s8', '('     ] = ('shift', 's4')
actionTable['s8', 'NUMBER'] = ('shift', 's5')

actionTable['s9', '('     ] = ('shift', 's4')
actionTable['s9', 'NUMBER'] = ('shift', 's5')

actionTable['s10', '+'] = ('shift', 's6')
actionTable['s10', '-'] = ('shift', 's7')
actionTable['s10', ')'] = ('shift', 's15')

actionTable['s11', '$'] = ('reduce', r1)
actionTable['s11', '+'] = ('reduce', r1)
actionTable['s11', '-'] = ('reduce', r1)
actionTable['s11', '*'] = ('shift', 's8')
actionTable['s11', '/'] = ('shift', 's9')
actionTable['s11', ')'] = ('reduce', r1)

actionTable['s12', '$'] = ('reduce', r2)
actionTable['s12', '+'] = ('reduce', r2)
actionTable['s12', '-'] = ('reduce', r2)
actionTable['s12', '*'] = ('shift', 's8')
actionTable['s12', '/'] = ('shift', 's9')
actionTable['s12', ')'] = ('reduce', r2)

actionTable['s13', '$'] = ('reduce', r4)
actionTable['s13', '+'] = ('reduce', r4)
actionTable['s13', '-'] = ('reduce', r4)
actionTable['s13', '*'] = ('reduce', r4)
actionTable['s13', '/'] = ('reduce', r4)
actionTable['s13', ')'] = ('reduce', r4)

actionTable['s14', '$'] = ('reduce', r5)
actionTable['s14', '+'] = ('reduce', r5)
actionTable['s14', '-'] = ('reduce', r5)
actionTable['s14', '*'] = ('reduce', r5)
actionTable['s14', '/'] = ('reduce', r5)
actionTable['s14', ')'] = ('reduce', r5)

actionTable['s15', '$'] = ('reduce', r7)
actionTable['s15', '+'] = ('reduce', r7)
actionTable['s15', '-'] = ('reduce', r7)
actionTable['s15', '*'] = ('reduce', r7)
actionTable['s15', '/'] = ('reduce', r7)
actionTable['s15', ')'] = ('reduce', r7)

## The Goto Table

Below is the definition of the *goto table*.  It encodes the function
$$\texttt{goto}: Q \times V \rightarrow Q.$$
After the parser has reduced the symbol stack with a rule $A \rightarrow \beta$ and removed the states that
belong to $\beta$ from the stack of states, it pushes the state $\texttt{goto}(q, A)$ onto the stack of states,
where $q$ is the state that is now on top of the stack of states.  The table only needs entries for those states
that contain a marked rule where the dot is in front of a variable, i.e. for the states `s0`, `s4`, `s6`, `s7`,
`s8`, and `s9`.

In [ ]:
gotoTable: dict[tuple[State, Variable], State]   = {}

gotoTable['s0', 'e'] = 's1'
gotoTable['s0', 'p'] = 's2'
gotoTable['s0', 'f'] = 's3'

gotoTable['s4', 'e'] = 's10'
gotoTable['s4', 'p'] = 's2'
gotoTable['s4', 'f'] = 's3'

gotoTable['s6', 'p'] = 's11'
gotoTable['s6', 'f'] = 's3'

gotoTable['s7', 'p'] = 's12'
gotoTable['s7', 'f'] = 's3'

gotoTable['s8', 'f'] = 's13'
gotoTable['s9', 'f'] = 's14'

## The State Table

Finally, we define the *state table*.  It maps every state to the list of its marked rules.  The parser does
not need this table.  It is only used for pretty printing:  The notebook
`02-Shift-Reduce-Parser.ipynb` prints the marked rules of the current state in every step, which gives us a clue
about the information that is stored in the different states.

Although a state is a set of marked rules, we store it as a list, since the order of the marked rules matters:
They are listed in the order in which they are added to the state, i.e. first the marked rules from which the
closure is computed and then the marked rules that are added by the closure.  The states are computed in a
systematic order that depends on this order:  For every state, the symbols after the dot are processed in the order
in which they occur first.  If we stored the marked rules in a set, *Python* would list them in an order that
changes from one run to the next, since it randomizes the hash values of strings.

The start state `s0` is the closure of the marked rule `ŝ → • e $`, where `ŝ` is the new start symbol that is
added to the grammar.  All other states are computed with the function $\texttt{goto}$.  This computation is
shown in the section on SLR parsers in the lecture notes.  Tokens are written in double quotes, e.g. `"+"`.

In [ ]:
stateTable: dict[State, list[MarkedRule]] = {}

stateTable['s0']  = [ 'ŝ → • e $', 
                      'e → • e "+" p', 'e → • e "-" p', 'e → • p', 
                      'p → • p "*" f', 'p → • p "/" f', 'p → • f', 
                      'f → • "(" e ")"', 'f → • NUMBER'
                    ]
stateTable['s1']  = [ 'ŝ → e • $', 'e → e • "+" p', 'e → e • "-" p' ]
stateTable['s2']  = [ 'e → p •', 'p → p • "*" f', 'p → p • "/" f' ] 
stateTable['s3']  = [ 'p → f •' ] 
stateTable['s4']  = [ 'f → "(" • e ")"', 
                      'e → • e "+" p', 'e → • e "-" p', 'e → • p', 
                      'p → • p "*" f', 'p → • p "/" f', 'p → • f', 
                      'f → • "(" e ")"', 'f → • NUMBER'
                    ]
stateTable['s5']  = [ 'f → NUMBER •' ] 
stateTable['s6']  = [ 'e → e "+" • p',
                      'p → • p "*" f', 'p → • p "/" f', 'p → • f', 
                      'f → • "(" e ")"', 'f → • NUMBER'
                    ]
stateTable['s7']  = [ 'e → e "-" • p',
                      'p → • p "*" f', 'p → • p "/" f', 'p → • f', 
                      'f → • "(" e ")"', 'f → • NUMBER'
                    ]
stateTable['s8']  = [ 'p → p "*" • f', 'f → • "(" e ")"', 'f → • NUMBER' ] 
stateTable['s9']  = [ 'p → p "/" • f', 'f → • "(" e ")"', 'f → • NUMBER' ] 
stateTable['s10'] = [ 'f → "(" e • ")"', 'e → e • "+" p', 'e → e • "-" p' ]
stateTable['s11'] = [ 'e → e "+" p •', 'p → p • "*" f', 'p → p • "/" f' ]
stateTable['s12'] = [ 'e → e "-" p •', 'p → p • "*" f', 'p → p • "/" f' ]
stateTable['s13'] = [ 'p → p "*" f •' ] 
stateTable['s14'] = [ 'p → p "/" f •' ] 
stateTable['s15'] = [ 'f → "(" e ")" •' ]